# 🏰 Emberfall NPC — GPU distillation (self-play conversations)

Generates the *responsive* training dataset for the Emberfall NPC fine-tune: a
teacher model plays the NPC (exact game persona, STATE-conditioned) while a
player-simulator plays an adventurer with randomized goals — so every reply
depends on what was actually said. **Conversations advance in lockstep GPU
batches** (auto-scaled to your GPU: batch 64 on an A100, 16 on a T4) —
~1500 conversations in **~5-15 min on an A100**, under an hour on a T4.

Output: `emberfall_npc_distilled.jsonl` — feed it to `Emberfall_NPC_finetune_1.2b*.ipynb`.

**Runtime → change runtime type → GPU.** Resumable: re-running the generation
cell continues toward TARGET (output appends per conversation).

## 1 · Install + load the teacher

In [ ]:
%pip -q uninstall -y torchao
%pip -q install "transformers>=4.57,<5" accelerate sentencepiece
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
assert torch.cuda.is_available(), "No GPU! Runtime -> change runtime type -> GPU."
GPU = torch.cuda.get_device_properties(0)
VRAM_GB = GPU.total_memory / 1e9
print(f"GPU: {GPU.name} ({VRAM_GB:.0f} GB)")

TEACHER = "unsloth/Llama-3.2-3B-Instruct"        # ungated mirror; best engagement in our shoot-out
# TEACHER = "unsloth/Meta-Llama-3.1-8B-Instruct" # A100 option: smarter teacher (ungated mirror)
# TEACHER = "LiquidAI/LFM2-2.6B"                 # alternative: best STATE fidelity

TARGET = 1500          # total conversations in the output file
# batch auto-scales with VRAM: A100-40GB -> 64, L4/A100-lite -> 32, T4 -> 16
BATCH  = 64 if VRAM_GB > 30 else 32 if VRAM_GB > 20 else 16
SEED   = 23
OUT    = "/content/emberfall_npc_distilled.jsonl"

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tok = AutoTokenizer.from_pretrained(TEACHER)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"                    # required for batched decoder-only generate
model = AutoModelForCausalLM.from_pretrained(TEACHER, torch_dtype=DTYPE, device_map="auto")
model.eval()
print("teacher:", TEACHER, "| dtype:", DTYPE, "| batch:", BATCH)

## 2 · Emberfall persona machinery *(embedded verbatim from `gen_npc_data_v2.py` — the training schema matches the game exactly)*

In [ ]:
#!/usr/bin/env python3
"""Generate STATE-conditioned Emberfall NPC SFT data where NPCs REACT to the
player's live condition — stink, wounds, combat renown, and how heavily armed
they are — on top of the existing location/time/weather/mood/relationship/shop
schema. Output matches js/gameplay/npc-chat.js → npcPersona so training == inference.

Usage: python gen_npc_data_v2.py --n 4500 --out emberfall_npc_reactions.jsonl [--seed 11]
"""
import argparse, json, random, hashlib

# role -> (skill, location, voice, traits, stock)  — mirrors the game's NPC_ROLE_INFO
ROLE_INFO = {
 "fisher":("Fishing","harbour","weathered","patient, plain-spoken","hooks, line, bait, smoked eel"),
 "farmer":("Farming","lower fields","rough","hardy, blunt","seed, onions, beans, oats"),
 "miller":("Milling","mill","steady","precise, tired","wheat flour, barley meal, oat flour, bran"),
 "baker":("Baking","bakehouse","warm","cheerful, floury","bread, oatcakes, meat pies, berry tarts"),
 "cook":("Cooking","inn kitchen","boisterous","busy, generous","stew, roast meat, fish pie, whitebait fritters"),
 "brewer":("Brewing","brew shed","mellow","easygoing, proud","small ale, cider, malt, yeast"),
 "beekeeper":("Beekeeping","orchard hives","soft","calm, careful","honey, beeswax, candles, propolis"),
 "herbalist":("Foraging","herb garden","quiet","knowing, wry","dried herbs, roots, flowers, salves"),
 "apothecary":("Potionmaking","apothecary","precise","clinical, dry","empty vials, tonics, salves, dried ingredients"),
 "woodcutter":("Woodcutting","forest edge","gruff","strong, direct","logs, kindling, handles, split timber"),
 "carpenter":("Carpentry","work yard","craftsmanlike","measured, exact","planks, pegs, tool handles, simple furniture"),
 "fletcher":("Fletching","fletcher's bench","focused","steady, terse","arrows, shafts, bowstrings, feathers"),
 "miner":("Ore-mining","mine mouth","rough","tough, grim","iron ore, copper, coal, a lump of tin"),
 "blacksmith":("Smithing","the forge","gruff","solid, proud","nails, hinges, axe heads, blades"),
 "armourer":("Smithing","armoury","stern","exacting, blunt","helms, mail rings, buckles, plate"),
 "mason":("Stone-mining","stoneyard","solid","slow, deliberate","cut stone, slate, lime, gravel"),
 "potter":("Pottery","kiln","calm","patient, tidy","pots, jugs, bowls, crocks"),
 "tanner":("Tanning","tanning racks","salt-dry","blunt, brisk","leather, hide, straps, pouches"),
 "weaver":("Weaving","loom-house","rhythmic","focused, wry","wool, cloth, thread, a warm cloak"),
 "dyer":("Dyeing","dye yard","colourful","lively, stained","dyed wool, woad, madder, fixed cloth"),
 "tailor":("Tailoring","tailor's shop","refined","neat, particular","tunics, cloaks, caps, mended garments"),
 "jeweller":("Jewelry","jeweller's bench","elegant","delicate, sharp-eyed","rings, amulets, cut gems, gold wire"),
 "hunter":("Hunting","the lodge","low","watchful, spare","furs, venison, snares, a bundle of arrows"),
 "forager":("Foraging","forest edge","wandering","curious, gentle","berries, mushrooms, nuts, wild herbs"),
 "sailor":("Sailing","the docks","salt-dry","hearty, rough","rope, salt, canvas, a sea-charm"),
 "ferryman":("Sailing","the ferry landing","slow","laconic, dry","a crossing, a pole, a bailing pail"),
 "merchant":("Trading","the market stall","smooth","shrewd, chatty","tools, cloth, spices, trinkets"),
 "innkeeper":("Cooking","the tavern","welcoming","jovial, quick","a mug of ale, a hot meal, a bed for the night"),
 "guard":("Melee","the gate","strict","watchful, curt","none listed"),
 "healer":("Potionmaking","the sickhouse","gentle","kind, tired","bandages, tonics, poultices, clean water"),
 "dockworker":("Sailing","the docks","rough-friendly","burly, blunt","crates, rope, a hand with your load"),
 "stablehand":("Husbandry","the stables","soft-spoken","quiet, steady","feed, tack, a stall for the night"),
 "shepherd":("Husbandry","the pasture","mellow","calm, weather-worn","wool, mutton, cheese, a fleece"),
 "orchardist":("Farming","the orchard","warm","patient, sunny","apples, pears, cider, dried fruit"),
 "boatbuilder":("Shipwrighting","the boatyard","craftsmanlike","careful, proud","planks, pitch, oars, a small skiff"),
 "cartwright":("Carpentry","the cart shed","mechanical","practical, terse","wheels, axles, cart parts, grease"),
 "scribe":("Bookbinding","the scriptorium","academic","precise, formal","ink, parchment, quills, a copied page"),
 "storyteller":("Storytelling","the tavern","lyrical","colourful, warm","a tale, a song, a rumour or two"),
 "elder":("Lore","the meeting hall","reflective","dignified, slow","none listed"),
 "mapmaker":("Cartography","the map room","measured","meticulous, calm","maps, charts, a compass, ink"),
}
ROLES = list(ROLE_INFO)

TIMES   = ["dawn","morning","midday","afternoon","dusk","evening"]
WEATHER = ["clear","windy","cold","warm","misty","light rain","steady rain","storm threatening"]
MOODS   = ["busy","calm","tired","curious","concerned","pleased","cheerful","irritated"]
RELS    = [("hostile",-55),("wary",-25),("unfamiliar",0),("neutral",15),("friendly",35),("trusted",60),("close",80)]
SKILLS  = ["novice","beginner","competent","skilled","expert"]
STINK   = ["fresh","whiffy","rank","reeking"]
HEALTH  = ["hale","hurt","wounded","bloodied","near death"]
COMBAT  = ["green","seasoned","veteran","renowned","legendary"]
ARMED   = ["unarmed","armed","armed and armoured"]
NAMES = ["Mira","Bram","Ketha","Sten","Torin","Wrenna","Alden","Peca","Ysolde","Garrick","Nessa","Corin",
 "Halda","Dob","Elga","Fenn","Marga","Osric","Petra","Rulf","Selam","Tupa","Karim","Nezahual","Wilf",
 "Bryn","Cara","Doran","Efa","Gethin","Isolde","Joss","Luca","Odd","Rhian","Sable","Tamsin","Udo","Vesna","Yorick"]

GREETINGS = ["Hello there!","Hello.","Good day to you.","Evening.","Well met.","Oi, morning.","Greetings.","Hail."]
QUESTIONS = ["Any news lately?","What's the word around here?","Any trouble on the roads?","What are you selling?",
 "How's business?","What do you do around here?","Which way to the market?","Got any advice for a traveller?",
 "Anything happening in town?","Fine weather, isn't it?"]

def hostile(rel): return rel[1] < 0
def friendly(rel): return rel[1] >= 35

# ── reaction banks (player condition drives the reply) ───────────────────────
def react_stink(level, rel, role, rng):
    if level == "reeking":
        base = ["Gods above — what died on you? Stand back, stand back!",
                "You smell like a midden in high summer. Away, before I'm sick!",
                "Faugh! That stench could curdle milk. Wash before you come near.",
                "By the gods, the reek of you! Downwind, if you please."]
        if role in ("merchant","baker","cook","innkeeper","tailor","jeweller","apothecary"):
            base += ["You'll not bring that stench into my shop — go wash, then we'll talk.",
                     "I'll bar the door before I let that smell in. Find a river, friend."]
        if friendly(rel): base += ["Oh, love, you honestly reek — go and bathe, for your own sake."]
        return rng.choice(base)
    # rank
    base = ["Phew — you could do with a wash, friend.","Is that you reeking, or have the pigs got loose?",
            "You've a powerful whiff about you. A river's that way.","Mind the nose — you've been at some sweaty work, eh?"]
    if hostile(rel): base += ["You stink. Keep your distance."]
    return rng.choice(base)

def react_health(level, rel, role, rng):
    healer = role in ("healer","apothecary","herbalist")
    if level == "near death":
        base = ["Gods, you're white as a shroud and bleeding! Sit, before you drop.",
                "You're half-dead on your feet — someone fetch the healer, quick!",
                "Steady now — you'll not last like that. Down, and let me look."]
        if healer: base = ["Lie down, now — you're near gone. Drink this, and don't argue.",
                           "You're bleeding out, friend. Hush and let me work."]
        return rng.choice(base)
    if level == "bloodied":
        base = ["You're bleeding badly — best see to that wound.","That's a nasty gash. There's a healer up the lane.",
                "Gods, look at the state of you. Bind that before it festers."]
        if healer: base = ["Sit — that wound wants cleaning and binding before it turns.",
                           "Nasty. Hold still, I've a poultice for that."]
        return rng.choice(base)
    # wounded / hurt
    base = ["You've taken a knock, I see. Careful out there.","Those cuts want tending — mind they don't fester.",
            "Rough day on the road, by the look of you."]
    if healer: base += ["Come by the sickhouse if those hurts worsen."]
    return rng.choice(base)

def react_combat(level, rel, role, rng):
    if level == "legendary":
        if hostile(rel): return rng.choice(["A blade of your renown, here? Keep it sheathed in my town.",
            "I know your name, and I want no part of your quarrels. Move along."])
        if friendly(rel): return rng.choice(["A warrior of your fame, at my table? The honour's mine.",
            "They sing of you in the tavern, you know. Anything you need, just ask.",
            "Well now — a proper hero in Newhaven. You're welcome here, always."])
        return rng.choice(["I've heard the tales of you. We want no trouble, only quiet.",
            "A fighter of your standing? Gods keep us — mind the peace here."])
    if level == "renowned":
        return rng.choice(["You've the bearing of someone who's seen real fights.",
            "Word travels — you've made a name with that blade.","A seasoned hand, plainly. The roads fear you more than you them."])
    if level == "green":
        base = ["You look barely blooded, lad. Mind the roads.","Green as spring grass, you are — keep close to the village a while.",
                "New to the sword, eh? Best not stray far past the walls yet."]
        if friendly(rel): base += ["Don't take it hard — we all started raw. Stay careful out there."]
        return rng.choice(base)
    return None  # seasoned/veteran: no special comment

def react_armed(level, rel, role, rng):
    guard = role in ("guard","armourer","blacksmith")
    base = ["All that steel — expecting a war, are you?","You're armed to the teeth. We're peaceful folk here.",
            "That's a lot of iron to carry into a quiet town."]
    if guard: base += ["Keep that blade sheathed past the gate, warrior. Rules are rules.",
                       "Armed and armoured, eh? Nothing I've not seen. Behave, and we've no quarrel."]
    if hostile(rel): base += ["Hand off your sword in my sight, if you don't mind."]
    if friendly(rel): base += ["Kitted for a fight, I see — hope you'll not need it round here."]
    return rng.choice(base)

# ── neutral chatter (no salient player condition) ────────────────────────────
RUMORS = ["strange lights out past the ridge","something's been taking sheep near the marsh","the old forest road's gone quiet",
 "bandits at the ford again","a peddler was robbed near the crossroads","the river's high after all the rain",
 "the miller's been shorting folk on flour","queer tracks down by the mill"]
def neutral(intent, role, rel, rng):
    skill, loc, voice, traits, stock = ROLE_INFO[role]
    ware = None if stock == "none listed" else stock
    banks = {
      "greet":[f"Well met, traveller.", f"A new face — welcome. What brings you by?", f"Good day. Don't see many strangers at {loc}.",
               f"Morning. Word is {rng.choice(RUMORS)}."],
      "news":[f"Word is {rng.choice(RUMORS)}. Quiet enough otherwise.", f"Only that {rng.choice(RUMORS)} — you know how folk talk.",
              f"Not much, save {rng.choice(RUMORS)}."],
      "roads":[f"Keep to daylight and you'll be fine — it's after dark I'd worry.", f"Fair enough, but mind the crossroads.",
               f"I'd not travel alone past the ridge, myself."],
      "trade":[f"Got {ware} if you've the coin." if ware else "I've naught to sell — try the market stall.",
               f"Fresh {ware} today, best in the village." if ware else "Not my trade, selling. Ask a merchant."],
      "work":[f"Busy as ever — no rest for the likes of me.", f"Can't complain. {skill} keeps a roof overhead.",
              f"Same as always, dawn till dusk at {loc}."],
      "self":[f"Me? Just the local {role}, been at {loc} longer than I'll admit.", f"I mind {loc} and my own business, mostly."],
      "direct":[f"The market's up past {loc}, can't miss it.", f"Follow the lane to the square — the inn's got the painted sign.",
                f"Shrine's on the hill, forge by the gate."],
      "help":[f"Keep your blade sharp and don't trust a road after dark.", f"Stock up before you leave — supplies are dear in the wilds.",
              f"Talk to folk, buy what you can, and steer clear of the marsh."],
      "small":[f"Aye, the weather's been something. Hard on old bones.", f"Quiet today — quiet's good, means no trouble about.",
               f"Never a dull season here, truth be told."],
    }
    return rng.choice(banks.get(intent, banks["greet"]))

# ── salient-condition picker (priority order) ────────────────────────────────
def salient_driver(stink, health, combat, armed):
    if stink in ("rank","reeking"): return ("stink", stink)
    if health in ("wounded","bloodied","near death"): return ("health", health)
    if combat in ("green","renowned","legendary"): return ("combat", combat)
    if armed == "armed and armoured": return ("armed", armed)
    return None

def make_reply(driver, pstate, rel, role, intent, rng):
    stink, health, combat, armed = pstate
    if driver:
        kind, lvl = driver
        if kind == "stink":  return react_stink(lvl, rel, role, rng)
        if kind == "health": return react_health(lvl, rel, role, rng)
        if kind == "combat":
            r = react_combat(lvl, rel, role, rng)
            if r: return r
        if kind == "armed":  return react_armed(lvl, rel, role, rng)
    return neutral(intent, role, rel, rng)

def persona(name, role, rel, pstate, rng):
    skill, loc, voice, traits, stock = ROLE_INFO[role]
    stink, health, combat, armed = pstate
    tod = rng.choice(TIMES); daytime = tod != "evening"
    shop_open = (stock != "none listed") and daytime
    st = "; ".join([
        f"location={loc}", f"time={tod}", f"weather={rng.choice(WEATHER)}", f"mood={rng.choice(MOODS)}",
        f"relationship={rel[0]}({rel[1]})", "quest=none", f"player_skill={rng.choice(SKILLS)}",
        f"player_stink={stink}", f"player_health={health}", f"player_combat={combat}", f"player_armed={armed}",
        f"shop_open={'true' if shop_open else 'false'}", f"stock={stock if shop_open else 'none listed'}", "known_fact=none.",
    ])
    sysp = (f"You are {name}, an Emberfall {role}. Voice={voice}; traits={traits}. Reply naturally in-world in "
            f"1-3 short sentences. Stay in character. Use STATE as the only source for player history, quest status, "
            f"shop state, stock, and current conditions. React to the player's condition: recoil from a foul stink, "
            f"show concern at wounds, defer to or dismiss their combat renown, note if they are heavily armed. Never "
            f"invent quest completion, inventory, prices, memories, or map routes. You know ordinary {skill} work and "
            f"local life. STATE: {st}")
    return sysp

def pick_player_state(rng, force_reaction):
    if force_reaction:
        # bias toward a salient condition so reactions are well represented
        kind = rng.choice(["stink","health","combat","armed"])
        stink = rng.choice(["rank","reeking"]) if kind=="stink" else rng.choice(["fresh","fresh","whiffy"])
        health = rng.choice(["wounded","bloodied","near death"]) if kind=="health" else rng.choice(["hale","hale","hurt"])
        combat = rng.choice(["green","renowned","legendary"]) if kind=="combat" else rng.choice(["seasoned","veteran"])
        armed = "armed and armoured" if kind=="armed" else rng.choice(["unarmed","armed","armed"])
    else:
        stink = rng.choices(STINK, weights=[70,20,7,3])[0]
        health = rng.choices(HEALTH, weights=[70,18,8,3,1])[0]
        combat = rng.choices(COMBAT, weights=[25,35,25,10,5])[0]
        armed = rng.choices(ARMED, weights=[25,55,20])[0]
    return (stink, health, combat, armed)


print("personas ready:", len(ROLE_INFO), "roles")

## 3 · Distillation engine (scenarios, scrubbing, grounding filter, conversation state machine)

In [ ]:
# ── distillation engine: scenario logic + quality gates (mirrors gen_npc_distill.py) ──
import json, random, re, time, torch

PLAYER_NAMES = ["Finn", "Ash", "Rowan", "Tama", "Isla", "Bren", "Kea", "Moss", "Aroha", "Silas"]
NPC_STYLE = (" Speak plainly in 1-3 short sentences, only words said aloud. No asterisks, no stage "
             "directions, no lists, no quotation marks.")
_BAD_RE = re.compile(r"STATE:|player_(stink|health|combat|armed|skill)|\bAI\b|language model", re.I)
_STINK_RE = re.compile(r"\b(stink|stench|reek|smell)\w*", re.I)
_WOUND_RE = re.compile(r"\b(wound|bleed|blood|injur|gash)\w*", re.I)
_QUOTE_RE = re.compile(r'^\s*["“‘\']+|["”’\']+\s*$')
_PREFIX_RE = re.compile(r"^\s*[A-Z][\w '\-]{0,24}:\s*")
_LIST_RE = re.compile(r"(?m)^\s*(?:[-*•]|\d+[.)])\s.*$")
_MD_RE = re.compile(r"[*_`#>]+")

def _clean(text):
    t = (text or "").strip()
    m = _LIST_RE.search(t)
    if m:
        t = t[:m.start()].strip()
        if t.endswith(":"):
            t = t[:t.rfind(".") + 1] if "." in t else ""
    t = _MD_RE.sub("", t)
    t = _PREFIX_RE.sub("", t, count=1)
    t = _QUOTE_RE.sub("", t)
    t = re.sub(r"\s+", " ", t).strip()
    if len(t) > 220:
        cut = t[:220]
        dot = max(cut.rfind(". "), cut.rfind("! "), cut.rfind("? "))
        t = (cut[:dot + 1] if dot > 60 else cut).strip()
    return t

def scrub(t):
    t = _clean(t)
    return re.sub(r"\s*\([^)]*\)", "", t).strip()

def grounded(reply, pstate, player_lines):
    stink, health, _c, _a = pstate
    said = " ".join(player_lines)
    if stink in ("fresh", "whiffy") and _STINK_RE.search(reply) and not _STINK_RE.search(said):
        return False
    if health in ("hale", "hurt") and _WOUND_RE.search(reply) and not (
            _WOUND_RE.search(said) or re.search(r"\b(stab|hurt|cut|fight|fought)\w*", said, re.I)):
        return False
    return True

def scenarios(rng, pname, role):
    stock = ROLE_INFO[role][4]
    off_stock = rng.choice(["a meat pie", "a sword", "a map of the isle", "a healing potion",
                            "a pair of boots", "some ale", "arrows", "a lantern"])
    return [
        (f"You want to buy {off_stock} - which they may not sell. If they don't have it, ask what they DO sell.",
         None, None, (2, 3)),
        ("You want local news or rumours. When they mention something, ask a follow-up question about that exact thing.",
         None, None, (2, 3)),
        ("Introduce yourself by name early. Later, casually test whether they remember your name.",
         f"Hello! I'm {pname} - good to meet you.", "Heh - do you even remember my name?", (3, 4)),
        ("You're hurt and want advice or help with your wounds.", None, None, (2, 3)),
        ("You reek from hard work. Apologise for the smell and try to do business anyway.", None, None, (2, 3)),
        ("You just survived a dangerous fight and want to tell someone about it.", None, None, (2, 3)),
        ("You're lost and need directions to somewhere in town.", None, None, (1, 2)),
        ("Make small talk about the weather, then ask them about their trade and how business is.", None, None, (2, 3)),
        ("Ask what they're selling and haggle a little over one item.", None, None, (2, 3)),
        (f"Ask the {role} a question about their craft that shows genuine curiosity.", None, None, (2, 3)),
    ]

def player_sim_prompt(pname, npc_name, role, goal, pstate):
    stink, health, combat, armed = pstate
    cond = []
    if stink in ("rank", "reeking"): cond.append("you smell awful from hard work")
    if health in ("wounded", "bloodied", "near death"): cond.append("you are visibly wounded")
    if combat in ("renowned", "legendary"): cond.append("you are a famous warrior")
    if armed == "armed and armoured": cond.append("you are heavily armed and armoured")
    cond_s = ("Your own condition (speak of it in FIRST person - it is you, not them): "
              + "; ".join(cond) + ". ") if cond else ""
    return (f"You are role-playing {pname}, a player adventurer in a medieval fantasy game, chatting with "
            f"{npc_name}, the local {role}. {cond_s}Your goal: {goal} Say ONE short casual line (under 25 "
            f"words) as {pname} - plain speech only, no narration, no asterisks, no quotes. React naturally "
            f"to what {npc_name} just said.")

ROLES = list(ROLE_INFO)

class Conv:
    """One self-play conversation as a state machine (stages: player -> npc -> ... -> done)."""
    def __init__(self, rng):
        self.role = rng.choice(ROLES); self.npc = rng.choice(NAMES); self.pname = rng.choice(PLAYER_NAMES)
        self.rel = rng.choice(RELS); self.pstate = pick_player_state(rng, rng.random() < 0.45)
        self.sys_game = persona(self.npc, self.role, self.rel, self.pstate, rng)
        self.sys_npc = self.sys_game + NPC_STYLE
        goal, opener, closer, (lo, hi) = rng.choice(scenarios(rng, self.pname, self.role))
        self.opener, self.closer = opener, closer
        self.n_ex = rng.randint(lo, hi)
        self.sim_sys = player_sim_prompt(self.pname, self.npc, self.role, goal, self.pstate)
        self.msgs = []; self.i = 0; self.stage = "player"; self.retries = 0; self.dead = False
    def scripted_line(self):
        if self.i == 0 and self.opener: return self.opener
        if self.i == self.n_ex - 1 and self.closer: return self.closer
        return None
    def build_messages(self):
        if self.stage == "player":
            sim = [{"role": "system", "content": self.sim_sys}]
            for m in self.msgs:
                sim.append({"role": "assistant" if m["role"] == "user" else "user", "content": m["content"]})
            if not self.msgs:
                sim.append({"role": "user", "content": f"({self.npc} looks up as you approach.)"})
            return sim
        return [{"role": "system", "content": self.sys_npc}] + self.msgs
    def accept(self, text):
        if self.stage == "player":
            line = scrub(text)
            if _BAD_RE.search(line) or len(line) < 2: self.dead = True; return
            self.msgs.append({"role": "user", "content": line}); self.stage = "npc"
        else:
            cand = scrub(text)
            pl = [m["content"] for m in self.msgs if m["role"] == "user"]
            if len(cand) >= 2 and not _BAD_RE.search(cand) and grounded(cand, self.pstate, pl):
                self.msgs.append({"role": "assistant", "content": cand})
                self.i += 1; self.stage = "player"
                if self.i >= self.n_ex: self.stage = "done"
            else:
                self.retries += 1
                if self.retries > 1: self.dead = True
    def record(self):
        return {"messages": [{"role": "system", "content": self.sys_game}] + self.msgs}
print("engine ready:", len(ROLES), "roles")

## 4 · Generate
Watch the rate line — an **A100 typically does 150-400 conversations/min** (T4:
30-60). Interrupt any time; re-run this cell to continue toward TARGET.

In [ ]:
# ── batched self-play: 16 conversations advance in lockstep GPU waves ──
def wave(convs, max_new):
    prompts = [tok.apply_chat_template(c.build_messages(), tokenize=False, add_generation_prompt=True)
               for c in convs]
    enc = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1536).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=True, temperature=0.85,
                             top_p=0.9, repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

import os
done = sum(1 for _ in open(OUT)) if os.path.exists(OUT) else 0
todo = max(0, TARGET - done)
print(f"have {done}, generating {todo} more -> {OUT}")
rng = random.Random(SEED + done * 31)
pool, written, failed, t0 = [], 0, 0, time.time()
out_f = open(OUT, "a")
while written < todo:
    while len(pool) < BATCH:
        pool.append(Conv(rng))
    # scripted player lines are trusted (no generation needed)
    for c in pool:
        if c.stage == "player" and c.scripted_line() is not None:
            c.msgs.append({"role": "user", "content": c.scripted_line()}); c.stage = "npc"
    gen_players = [c for c in pool if c.stage == "player"]
    if gen_players:
        for c, t in zip(gen_players, wave(gen_players, 48)): c.accept(t)
    npcs = [c for c in pool if c.stage == "npc" and not c.dead]
    if npcs:
        for c, t in zip(npcs, wave(npcs, 80)): c.accept(t)
    for c in list(pool):
        if c.dead:
            pool.remove(c); failed += 1
        elif c.stage == "done":
            pool.remove(c)
            out_f.write(json.dumps(c.record(), ensure_ascii=False) + "\n"); out_f.flush()
            written += 1
    if written and written % 25 < 2:
        rate = written / (time.time() - t0)
        print(f"  {done + written}/{TARGET} | {rate*60:.0f}/min | ETA {(todo-written)/max(rate,1e-9)/60:.0f} min "
              f"| {failed} rejected", flush=True)
out_f.close()
print(f"DONE: {done + written} conversations in {OUT} ({failed} rejected)")

## 5 · Inspect samples + quality stats

In [ ]:
import json, re, random
rows = [json.loads(l) for l in open(OUT)]
bad_paren = bad_ground = 0
for r in rows:
    s = r["messages"][0]["content"]
    stink = re.search(r"player_stink=([^;]+)", s).group(1)
    health = re.search(r"player_health=([^;]+)", s).group(1)
    said = " ".join(m["content"] for m in r["messages"] if m["role"] == "user")
    for m in r["messages"]:
        if m["role"] != "assistant": continue
        if "(" in m["content"]: bad_paren += 1
        if stink in ("fresh","whiffy") and _STINK_RE.search(m["content"]) and not _STINK_RE.search(said): bad_ground += 1
print(f"{len(rows)} conversations | parenthetical leaks: {bad_paren} | stink-grounding violations: {bad_ground}")
random.seed(1)
for r in random.sample(rows, min(3, len(rows))):
    print("\n---", r["messages"][0]["content"].split("Emberfall ")[1].split(".")[0])
    for m in r["messages"][1:]:
        print(f"  {'P' if m['role']=='user' else 'N'}: {m['content'][:150]}")

## 6 · Download the dataset
Then upload it to `Emberfall_NPC_finetune_1.2b*.ipynb` and train.

In [ ]:
from google.colab import files
files.download(OUT)